# Week 1 - VibeSpace3D Infrastructure Debug Notebook

This notebook helps validate the new multi-view data and correspondence scaffolding.

In [ ]:
from pathlib import Path
import json
import torch
from PIL import Image
import matplotlib.pyplot as plt

from src.data.multiview_dataset import MultiViewDataset, tensorize_multiview_sample
from src.correspondence.multiview_correspondence import mutual_nn_correspondence

In [ ]:
# Build a tiny local manifest from existing repo images for debugging
debug_manifest = Path('tmp/week1_debug_manifest.json')
debug_manifest.parent.mkdir(parents=True, exist_ok=True)

manifest = {
  'samples': [
    {
      'object_id': 'debug_object',
      'text': 'debug sample from existing images',
      'views': [
        {
          'image': 'images/00436_l.jpg',
          'K': [[500,0,256],[0,500,256],[0,0,1]],
          'T_wc': [[1,0,0,0],[0,1,0,0],[0,0,1,0],[0,0,0,1]]
        },
        {
          'image': 'images/00436_r.jpg',
          'K': [[500,0,256],[0,500,256],[0,0,1]],
          'T_wc': [[1,0,0,0.1],[0,1,0,0],[0,0,1,0],[0,0,0,1]]
        }
      ]
    }
  ]
}
debug_manifest.write_text(json.dumps(manifest, indent=2))
debug_manifest

In [ ]:
# Load dataset and inspect sample
dataset = MultiViewDataset(debug_manifest)
sample = dataset[0]
sample.object_id, len(sample.views), sample.text

In [ ]:
# Tensorize with a lightweight transform
from torchvision import transforms
transform = transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor()])
batch = tensorize_multiview_sample(sample, transform)
batch['images'].shape, batch['intrinsics'].shape, batch['poses_wc'].shape

In [ ]:
# Visualize the two debug views
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
for i in range(2):
    axes[i].imshow(batch['images'][i].permute(1,2,0))
    axes[i].set_title(f'View {i}')
    axes[i].axis('off')
plt.tight_layout()

In [ ]:
# Correspondence smoke test with synthetic token features
src = torch.randn(128, 64)
dst = src + 0.05 * torch.randn(128, 64)
corr = mutual_nn_correspondence(src, dst, min_confidence=0.1)
len(corr.src_indices), corr.confidences.mean().item()